# Mnemosyne Python SDK — Jupyter Notebook Quickstart

This notebook demonstrates how to use the Mnemosyne Python SDK to store and retrieve persistent memory for AI agents.

**Prerequisites:**
- Mnemosyne daemon running: `mnemosyne start`
- Python SDK installed: `pip install mnemosyne-sdk`

In [ ]:
import asyncio
from mnemosyne import MnemosyneClient

## 1. Connect to the Daemon

In [ ]:
# Create and connect
client = MnemosyneClient("localhost:50051")
await client.connect()

# Health check
status = await client.ping()
print(f"Daemon status: {status}")

## 2. Add Memories

In [ ]:
# Save facts that persist across sessions
mem1 = await client.add_memory(
    key="deploy-server",
    content="The deploy server is at node-3.internal",
    category="environment",
    priority=4,
)
print(f"Added: {mem1}")

mem2 = await client.add_memory(
    key="deployment-process",
    content="Use `make deploy` for local, `make deploy-studio` for Studio",
    category="project",
    priority=4,
)
print(f"Added: {mem2}")

mem3 = await client.add_memory(
    key="aj-pronouns",
    content="AJ prefers they/them pronouns",
    category="user",
    priority=5,
)
print(f"Added: {mem3}")

## 3. Get the Hot Block

The hot block is a priority-sorted context window — the most relevant memories for prompt injection.

In [ ]:
block = await client.get_hot_block(target="memory")
print(f"Hot block: {block.entry_count} entries, {block.char_count} chars")
print(f"\n{block.content}")

## 4. Search Memories

In [ ]:
results = await client.search_memories("deploy")
print(f"Found {len(results)} results for 'deploy':")
for m in results:
    print(f"  [{m.category}] {m.key}: {m.content[:60]}")

## 5. Session Management

In [ ]:
# Create a session
session = await client.create_session(
    agent_id="jupyter-notebook",
    source="jupyter",
    title="SDK Demo Session",
)
print(f"Created session: {session.id[:8]}... (title: {session.title})")

# Search past sessions
sessions = await client.list_sessions(limit=5)
print(f"\nRecent sessions:")
for s in sessions:
    print(f"  {s.id[:8]}... | {s.title} | {s.message_count} messages")

## 6. Update and Remove

In [ ]:
# Update a memory
updated = await client.update_memory(
    mem1.id,
    content="The deploy server moved to node-5.internal",
    priority=5,
)
print(f"Updated: {updated.key} → {updated.content}")

# Remove a memory
deleted = await client.remove_memory(mem3.id)
print(f"Deleted: {deleted}")

## 7. List All Memories

In [ ]:
all_mems = await client.list_memories()
print(f"Total memories: {len(all_mems)}")
for m in all_mems:
    print(f"  [{m.category}] {m.key}: {m.content[:50]} (priority={m.priority})")

## 8. Todos

In [ ]:
# Add a todo
todo = await client.add_todo(content="Review PR #42", priority=4)
print(f"Added todo: {todo}")

# Complete it
completed = await client.complete_todo(todo.id)
print(f"Completed: {completed}")

## 9. Cleanup

In [ ]:
# Close the connection
await client.close()
print("Connection closed.")

# Or use the context manager pattern:
async with MnemosyneClient("localhost:50051") as c:
    status = await c.ping()
    print(f"Reconnected: {status}")